# Deep Neural Network — L-Layer

> Based on Stanford CS230 — C1M4, Andrew Ng / DeepLearning.AI

Generalising a 2-layer network to $L$ layers requires careful bookkeeping of shapes and a clean loop over layers. This notebook builds the full forward + backward + update loop for any depth.

## Learning Objectives

1. Use general $L$-layer notation consistently
2. Count parameters and understand what controls model capacity
3. Implement a training loop for an $L$-layer network
4. Distinguish parameters (learned) from hyperparameters (chosen by you)

## Parameters vs Hyperparameters

| Category | Examples |
|---|---|
| **Parameters** (learned by GD) | $W^{[1]}, b^{[1]}, \ldots, W^{[L]}, b^{[L]}$ |
| **Hyperparameters** (chosen by you) | Learning rate $\alpha$, depth $L$, units per layer, activation choice, # iterations |

> Hyperparameters control the final parameters — tuning them is an empirical process (try, evaluate on dev set, iterate).


## Architecture Diagram — 4-Layer Network

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 700 240" width="700" height="240" style="font-family:DejaVu Sans,Arial,sans-serif;background:#f5f5f7;border-radius:8px">
  <!-- Layer x-positions and node counts -->
  <!-- L0 (input): x=60, 4 nodes -->
  <!-- L1: x=200, 5 nodes -->
  <!-- L2: x=340, 5 nodes -->
  <!-- L3: x=480, 4 nodes -->
  <!-- L4 (output): x=620, 1 node -->
  <defs>
    <marker id="a" markerWidth="6" markerHeight="6" refX="5" refY="3" orient="auto">
      <path d="M0,0 L0,6 L6,3 z" fill="#c8ccd4"/>
    </marker>
  </defs>

  <!-- Helper: connections between layers (drawn first so nodes appear on top) -->
  <!-- L0 (y: 50,90,130,170) → L1 (y: 30,70,110,150,190) -->
  <!-- L0 nodes -->
  <!-- connections L0→L1 -->
  <g stroke="#dde0e8" stroke-width="0.9">
    <line x1="60" y1="50"  x2="200" y2="30"/> <line x1="60" y1="50"  x2="200" y2="70"/>
    <line x1="60" y1="50"  x2="200" y2="110"/><line x1="60" y1="50"  x2="200" y2="150"/>
    <line x1="60" y1="50"  x2="200" y2="190"/>
    <line x1="60" y1="90"  x2="200" y2="30"/> <line x1="60" y1="90"  x2="200" y2="70"/>
    <line x1="60" y1="90"  x2="200" y2="110"/><line x1="60" y1="90"  x2="200" y2="150"/>
    <line x1="60" y1="90"  x2="200" y2="190"/>
    <line x1="60" y1="130" x2="200" y2="30"/> <line x1="60" y1="130" x2="200" y2="70"/>
    <line x1="60" y1="130" x2="200" y2="110"/><line x1="60" y1="130" x2="200" y2="150"/>
    <line x1="60" y1="130" x2="200" y2="190"/>
    <line x1="60" y1="170" x2="200" y2="30"/> <line x1="60" y1="170" x2="200" y2="70"/>
    <line x1="60" y1="170" x2="200" y2="110"/><line x1="60" y1="170" x2="200" y2="150"/>
    <line x1="60" y1="170" x2="200" y2="190"/>
  </g>
  <!-- connections L1→L2 -->
  <g stroke="#dde0e8" stroke-width="0.9">
    <line x1="200" y1="30"  x2="340" y2="30"/> <line x1="200" y1="30"  x2="340" y2="70"/>
    <line x1="200" y1="30"  x2="340" y2="110"/><line x1="200" y1="30"  x2="340" y2="150"/>
    <line x1="200" y1="30"  x2="340" y2="190"/>
    <line x1="200" y1="70"  x2="340" y2="30"/> <line x1="200" y1="70"  x2="340" y2="70"/>
    <line x1="200" y1="70"  x2="340" y2="110"/><line x1="200" y1="70"  x2="340" y2="150"/>
    <line x1="200" y1="70"  x2="340" y2="190"/>
    <line x1="200" y1="110" x2="340" y2="30"/> <line x1="200" y1="110" x2="340" y2="70"/>
    <line x1="200" y1="110" x2="340" y2="110"/><line x1="200" y1="110" x2="340" y2="150"/>
    <line x1="200" y1="110" x2="340" y2="190"/>
    <line x1="200" y1="150" x2="340" y2="30"/> <line x1="200" y1="150" x2="340" y2="70"/>
    <line x1="200" y1="150" x2="340" y2="110"/><line x1="200" y1="150" x2="340" y2="150"/>
    <line x1="200" y1="150" x2="340" y2="190"/>
    <line x1="200" y1="190" x2="340" y2="30"/> <line x1="200" y1="190" x2="340" y2="70"/>
    <line x1="200" y1="190" x2="340" y2="110"/><line x1="200" y1="190" x2="340" y2="150"/>
    <line x1="200" y1="190" x2="340" y2="190"/>
  </g>
  <!-- connections L2→L3 -->
  <g stroke="#dde0e8" stroke-width="0.9">
    <line x1="340" y1="30"  x2="480" y2="50"/> <line x1="340" y1="30"  x2="480" y2="90"/>
    <line x1="340" y1="30"  x2="480" y2="130"/><line x1="340" y1="30"  x2="480" y2="170"/>
    <line x1="340" y1="70"  x2="480" y2="50"/> <line x1="340" y1="70"  x2="480" y2="90"/>
    <line x1="340" y1="70"  x2="480" y2="130"/><line x1="340" y1="70"  x2="480" y2="170"/>
    <line x1="340" y1="110" x2="480" y2="50"/> <line x1="340" y1="110" x2="480" y2="90"/>
    <line x1="340" y1="110" x2="480" y2="130"/><line x1="340" y1="110" x2="480" y2="170"/>
    <line x1="340" y1="150" x2="480" y2="50"/> <line x1="340" y1="150" x2="480" y2="90"/>
    <line x1="340" y1="150" x2="480" y2="130"/><line x1="340" y1="150" x2="480" y2="170"/>
    <line x1="340" y1="190" x2="480" y2="50"/> <line x1="340" y1="190" x2="480" y2="90"/>
    <line x1="340" y1="190" x2="480" y2="130"/><line x1="340" y1="190" x2="480" y2="170"/>
  </g>
  <!-- connections L3→L4 -->
  <g stroke="#dde0e8" stroke-width="0.9">
    <line x1="480" y1="50"  x2="620" y2="110"/>
    <line x1="480" y1="90"  x2="620" y2="110"/>
    <line x1="480" y1="130" x2="620" y2="110"/>
    <line x1="480" y1="170" x2="620" y2="110"/>
  </g>

  <!-- Layer 0 — input -->
  <circle cx="60" cy="50"  r="16" fill="#5b9bd5" stroke="#3a7bbf" stroke-width="1.5"/>
  <circle cx="60" cy="90"  r="16" fill="#5b9bd5" stroke="#3a7bbf" stroke-width="1.5"/>
  <circle cx="60" cy="130" r="16" fill="#5b9bd5" stroke="#3a7bbf" stroke-width="1.5"/>
  <circle cx="60" cy="170" r="16" fill="#5b9bd5" stroke="#3a7bbf" stroke-width="1.5"/>
  <text x="60" y="54"  text-anchor="middle" fill="white" font-size="10">x&#x2081;</text>
  <text x="60" y="94"  text-anchor="middle" fill="white" font-size="10">x&#x2082;</text>
  <text x="60" y="134" text-anchor="middle" fill="white" font-size="10">x&#x2083;</text>
  <text x="60" y="174" text-anchor="middle" fill="white" font-size="10">x&#x2084;</text>

  <!-- Layer 1 -->
  <circle cx="200" cy="30"  r="16" fill="#7ecba1" stroke="#5aab81" stroke-width="1.5"/>
  <circle cx="200" cy="70"  r="16" fill="#7ecba1" stroke="#5aab81" stroke-width="1.5"/>
  <circle cx="200" cy="110" r="16" fill="#7ecba1" stroke="#5aab81" stroke-width="1.5"/>
  <circle cx="200" cy="150" r="16" fill="#7ecba1" stroke="#5aab81" stroke-width="1.5"/>
  <circle cx="200" cy="190" r="16" fill="#7ecba1" stroke="#5aab81" stroke-width="1.5"/>

  <!-- Layer 2 -->
  <circle cx="340" cy="30"  r="16" fill="#7ecba1" stroke="#5aab81" stroke-width="1.5"/>
  <circle cx="340" cy="70"  r="16" fill="#7ecba1" stroke="#5aab81" stroke-width="1.5"/>
  <circle cx="340" cy="110" r="16" fill="#7ecba1" stroke="#5aab81" stroke-width="1.5"/>
  <circle cx="340" cy="150" r="16" fill="#7ecba1" stroke="#5aab81" stroke-width="1.5"/>
  <circle cx="340" cy="190" r="16" fill="#7ecba1" stroke="#5aab81" stroke-width="1.5"/>

  <!-- Layer 3 -->
  <circle cx="480" cy="50"  r="16" fill="#c678dd" stroke="#9a4ab0" stroke-width="1.5"/>
  <circle cx="480" cy="90"  r="16" fill="#c678dd" stroke="#9a4ab0" stroke-width="1.5"/>
  <circle cx="480" cy="130" r="16" fill="#c678dd" stroke="#9a4ab0" stroke-width="1.5"/>
  <circle cx="480" cy="170" r="16" fill="#c678dd" stroke="#9a4ab0" stroke-width="1.5"/>

  <!-- Layer 4 — output -->
  <circle cx="620" cy="110" r="18" fill="#e05c5c" stroke="#c03a3a" stroke-width="1.5"/>
  <text x="620" y="114" text-anchor="middle" fill="white" font-size="11" font-weight="bold">&#x177;</text>

  <!-- Layer labels -->
  <text x="60"  y="218" text-anchor="middle" fill="#555" font-size="10">Layer 0</text>
  <text x="60"  y="230" text-anchor="middle" fill="#888" font-size="9">(input)</text>
  <text x="200" y="218" text-anchor="middle" fill="#555" font-size="10">Layer 1</text>
  <text x="200" y="230" text-anchor="middle" fill="#888" font-size="9">n&#x00B9;=5</text>
  <text x="340" y="218" text-anchor="middle" fill="#555" font-size="10">Layer 2</text>
  <text x="340" y="230" text-anchor="middle" fill="#888" font-size="9">n&#x00B2;=5</text>
  <text x="480" y="218" text-anchor="middle" fill="#555" font-size="10">Layer 3</text>
  <text x="480" y="230" text-anchor="middle" fill="#888" font-size="9">n&#x00B3;=4</text>
  <text x="620" y="140" text-anchor="middle" fill="#555" font-size="10">Layer 4</text>
  <text x="620" y="152" text-anchor="middle" fill="#888" font-size="9">(output)</text>
</svg>


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({
    'figure.facecolor': '#f5f5f7', 'axes.facecolor': '#ffffff',
    'axes.edgecolor': '#c8ccd4', 'axes.labelcolor': '#1a1d27',
    'axes.titlecolor': '#1a1d27', 'xtick.color': '#2a2e3a',
    'ytick.color': '#2a2e3a', 'grid.color': '#e0e3ea',
    'grid.linestyle': '--', 'grid.alpha': 0.5,
    'text.color': '#1a1d27', 'font.family': 'DejaVu Sans',
    'axes.titlesize': 13, 'axes.labelsize': 11,
    'legend.facecolor': '#ffffff', 'legend.edgecolor': '#c8ccd4',
    'figure.dpi': 110,
})
P = ['#5b9bd5', '#e05c5c', '#f4b942', '#7ecba1', '#56b6c2', '#c678dd']

# -------------------------------------------------------------------
# General L-layer neural network implementation
# -------------------------------------------------------------------
def sigmoid(z):   return 1 / (1 + np.exp(-z))
def relu(z):      return np.maximum(0, z)

def init_params_he(dims, seed=1):
    np.random.seed(seed)
    p = {}
    for l in range(1, len(dims)):
        p[f'W{l}'] = np.random.randn(dims[l], dims[l-1]) * np.sqrt(2 / dims[l-1])
        p[f'b{l}'] = np.zeros((dims[l], 1))
    return p

def forward_L(X, params, L):
    caches, A = [], X
    for l in range(1, L + 1):
        W, b = params[f'W{l}'], params[f'b{l}']
        Z = W @ A + b
        A_prev = A
        A = relu(Z) if l < L else sigmoid(Z)
        caches.append((A_prev, W, b, Z, A))
    return A, caches

def cost_bce(AL, Y):
    return -np.mean(Y * np.log(AL + 1e-8) + (1-Y) * np.log(1-AL + 1e-8))

def backward_L(caches, params, Y, L):
    grads = {}
    m = Y.shape[1]
    # output layer
    _, W, b, Z, A = caches[-1]
    dA = -(Y/(A+1e-8) - (1-Y)/(1-A+1e-8))
    dZ = dA * A * (1 - A)  # sigmoid deriv
    A_prev = caches[-2][4] if L > 1 else caches[0][0]
    grads[f'dW{L}'] = (dZ @ A_prev.T) / m
    grads[f'db{L}'] = np.mean(dZ, axis=1, keepdims=True)
    dA_prev = params[f'W{L}'].T @ dZ

    for l in reversed(range(1, L)):
        A_prev_l, W_l, b_l, Z_l, A_l = caches[l-1]
        dZ_l = dA_prev * (Z_l > 0).astype(float)
        grads[f'dW{l}'] = (dZ_l @ A_prev_l.T) / m
        grads[f'db{l}'] = np.mean(dZ_l, axis=1, keepdims=True)
        dA_prev = params[f'W{l}'].T @ dZ_l
    return grads

def train(X, Y, dims, alpha=0.05, n_iter=3000, seed=1):
    L = len(dims) - 1
    params = init_params_he(dims, seed=seed)
    costs = []
    for i in range(n_iter):
        AL, caches = forward_L(X, params, L)
        costs.append(cost_bce(AL, Y))
        grads = backward_L(caches, params, Y, L)
        for l in range(1, L+1):
            params[f'W{l}'] -= alpha * grads[f'dW{l}']
            params[f'b{l}'] -= alpha * grads[f'db{l}']
    return params, costs

# ---- Parameter count ----
def count_params(dims):
    total = 0
    rows = []
    for l in range(1, len(dims)):
        w = dims[l] * dims[l-1]
        b = dims[l]
        rows.append((l, dims[l-1], dims[l], w, b, w+b))
        total += w + b
    print(f"{'Layer':>5} {'n[l-1]':>7} {'n[l]':>6} {'W':>8} {'b':>6} {'Total':>8}")
    print("-" * 48)
    for r in rows:
        print(f"{r[0]:>5} {r[1]:>7} {r[2]:>6} {r[3]:>8} {r[4]:>6} {r[5]:>8}")
    print("-" * 48)
    print(f"{'Grand total':>34} {total:>8}")
    return total

dims = [12, 20, 15, 10, 5, 1]
print("Parameter count for network:", dims)
count_params(dims)

# ---- Compare depth vs performance ----
np.random.seed(42)
m = 400
X = np.vstack([np.random.randn(2, m//2) + [[3], [3]],
               np.random.randn(2, m//2) + [[-3], [-3]]]).T.T
Y = np.array([1]*(m//2) + [0]*(m//2)).reshape(1, -1)

configs = {
    '1 hidden (4 units)':   [2, 4, 1],
    '2 hidden (8,4)':       [2, 8, 4, 1],
    '3 hidden (16,8,4)':    [2, 16, 8, 4, 1],
    '4 hidden (32,16,8,4)': [2, 32, 16, 8, 4, 1],
}

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Effect of Depth on Learning', fontsize=13, fontweight='bold')

final_costs = {}
for i, (label, dims) in enumerate(configs.items()):
    p, costs = train(X, Y, dims, alpha=0.1, n_iter=2000)
    axes[0].plot(costs, color=P[i], lw=2, label=label)
    final_costs[label] = costs[-1]

axes[0].set_xlabel('Iteration')
axes[0].set_ylabel('Cost J')
axes[0].set_title('Training Cost by Depth')
axes[0].legend(fontsize=8)
axes[0].grid(True)

labels = list(final_costs.keys())
vals   = list(final_costs.values())
bars = axes[1].bar(range(len(labels)), vals, color=P[:len(labels)], alpha=0.85)
axes[1].set_xticks(range(len(labels)))
axes[1].set_xticklabels([l.split(' (')[0] for l in labels], rotation=15, fontsize=9)
axes[1].set_ylabel('Final Cost')
axes[1].set_title('Final Cost Comparison')
axes[1].grid(True, axis='y')
for bar, v in zip(bars, vals):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
                 f'{v:.4f}', ha='center', fontsize=9)

plt.tight_layout()
plt.show()
